# 06 · Vegetation, Mixed Materials & Moisture Inversion

Reproduces:
- **Fig. 9** — Polarimetric phase between HH and VV channels vs time
  (De Zan 2014): expected to be near-zero for bare soil (Fresnel contribution only).
- **Fig. 10** — Moisture inversion from phase triplets + coherence magnitudes
  (De Zan 2014).
- **Fig. 11** — Moisture inversion from phase triplets only (De Zan 2014).

Also demonstrates how a vegetation/mixed-material component breaks the model
and inflates the polarimetric phase.

---
**References**
- De Zan, F., Parizzi, A., Prats-Iraola, P., & López-Dekker, P. (2014).
  *A SAR interferometric model for soil moisture.*
  IEEE Transactions on Geoscience and Remote Sensing, 52(1), 418–425.
  https://doi.org/10.1109/TGRS.2013.2241069
- Zheng, Y., & Fattahi, H. (2026).
  *Modeling, prediction, and retrieval of surface soil moisture from InSAR closure phase.*
  Remote Sensing of Environment, 333, 115104.
  https://doi.org/10.1016/j.rse.2025.115104

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from navasar.coherence import interferometric_coherence, fresnel_phase, phase_triplet

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
KW = dict(freq_ghz=1.4, theta_inc_deg=45.0, sand=0.51, clay=0.13)

## AGRISAR 2006 acquisition series

In [ ]:
doy    = np.array([131, 144, 157, 164, 172, 186, 193])
labels = ['May-11','May-24','Jun-06','Jun-13','Jun-21','Jul-05','Jul-12']
# True moisture (field 101, in-situ, Fig. 12 of De Zan 2014)
mv_true = np.array([0.28, 0.22, 0.18, 0.15, 0.12, 0.10, 0.09])
n = len(mv_true)

## Fig. 9 — Polarimetric phase HH–VV (De Zan 2014)

For bare soil the Fresnel contribution is tiny; vegetation inflates it.

In [ ]:
# Bare soil: polarimetric phase = phase(tau_TE) - phase(tau_TM)
phase_te, phase_tm = fresnel_phase(mv_true, **KW)
pol_phase_bare = phase_te - phase_tm

# Vegetation contamination: add random phase offset growing with time
rng = np.random.default_rng(7)
veg_growth = np.linspace(0, 1, n)**2
pol_phase_veg = pol_phase_bare + veg_growth * rng.normal(0, 8, n)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(doy, pol_phase_bare, 'b-o', ms=6, label='Bare soil (model)')
ax.plot(doy, pol_phase_veg,  'r-s', ms=6, label='With vegetation noise')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(doy); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('HH–VV polarimetric phase [deg]')
ax.set_title('Polarimetric phase HH–VV\n(De Zan 2014, Fig. 9)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/fig09_polarimetric_phase.png', dpi=150)
plt.show()

## Moisture inversion — forward model

Minimise the figure of merit (De Zan 2014, Eq. 16):

$$F = \sum_{n>m>k} |\phi'_{n,m,k} - \phi_{n,m,k}|^2 + \sum_{n,k} |\gamma'_{n,k} - \gamma_{n,k}|^2$$

In [ ]:
from itertools import combinations

# Simulate "observed" quantities from true moisture
obs_triplets = {}
obs_coh = {}
for i, j, k in combinations(range(n), 3):
    obs_triplets[(i,j,k)] = phase_triplet(mv_true[i], mv_true[j], mv_true[k], **KW)
for i, j in combinations(range(n), 2):
    obs_coh[(i,j)] = abs(interferometric_coherence(mv_true[i], mv_true[j], **KW))

def figure_of_merit(mv_vec, use_coh=True):
    """De Zan 2014 Eq. 16."""
    F = 0.0
    for (i,j,k), phi_obs in obs_triplets.items():
        phi_pred = phase_triplet(mv_vec[i], mv_vec[j], mv_vec[k], **KW)
        diff = phi_obs - phi_pred
        diff = (diff + 180) % 360 - 180   # wrap
        F += (diff / 180)**2
    if use_coh:
        for (i,j), g_obs in obs_coh.items():
            g_pred = abs(interferometric_coherence(mv_vec[i], mv_vec[j], **KW))
            F += (g_obs - g_pred)**2
    return F

# First moisture value fixed at 10% (De Zan 2014)
mv0 = np.full(n, 0.15)
mv0[0] = 0.10

def invert(use_coh):
    bounds = [(0.0, 0.5)] * n
    bounds[0] = (0.10, 0.10)   # fix first value
    res = minimize(figure_of_merit, mv0, args=(use_coh,),
                   method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 500, 'ftol': 1e-12})
    return res.x

mv_inv_both    = invert(use_coh=True)
mv_inv_triplet = invert(use_coh=False)
print('Inversion done.')

## Fig. 10 — Inversion from triplets + coherences (De Zan 2014)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(doy, mv_true,      'k-o',  ms=6, label='In-situ (truth)')
ax.plot(doy, mv_inv_both,  'b--s', ms=6, label='Inverted (triplets + |γ|)')
ax.set_xticks(doy); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Volumetric moisture $m_v$')
ax.set_title('Moisture inversion — triplets + coherence magnitudes\n(De Zan 2014, Fig. 10)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/fig10_inversion_triplets_coh.png', dpi=150)
plt.show()

## Fig. 11 — Inversion from triplets only (De Zan 2014)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(doy, mv_true,         'k-o',  ms=6, label='In-situ (truth)')
ax.plot(doy, mv_inv_triplet,  'r--^', ms=6, label='Inverted (triplets only)')
ax.set_xticks(doy); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Volumetric moisture $m_v$')
ax.set_title('Moisture inversion — phase triplets only\n(De Zan 2014, Fig. 11)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/fig11_inversion_triplets_only.png', dpi=150)
plt.show()

## Vegetation effect: model breakdown

Adding a vegetation coherence term shows how the model loses validity
when crops grow (July onwards in AGRISAR 2006).

In [ ]:
# Vegetation coherence decays exponentially with time
gamma_veg = np.exp(-np.arange(n) / 3.0)   # fast decorrelation

# Total coherence = soil coherence * vegetation coherence
fig, ax = plt.subplots(figsize=(7, 4))
for master_idx in [0, 2, 4]:
    slave_idx = [s for s in range(n) if s != master_idx]
    coh_soil = np.abs(interferometric_coherence(
        mv_true[master_idx], mv_true[slave_idx], **KW))
    dt = np.abs(np.array(slave_idx) - master_idx)
    coh_total = coh_soil * np.exp(-dt / 4.0)
    ax.plot(doy[slave_idx], coh_total, '-o', ms=5,
            label=f'Master {labels[master_idx]}')

ax.set_xticks(doy); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('|γ| (soil × vegetation)')
ax.set_title('Coherence with vegetation decorrelation\n(model validity breaks in July)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/fig_vegetation_coherence.png', dpi=150)
plt.show()